# Alert Mesh — how a warning reaches people with no signal

Alert Mesh is an emergency warning app for floods and bushfires in remote Australia.
Official warnings are **signed**, sent to phones that have a connection, and then
**passed from phone to phone over Bluetooth**, so they also reach people with no signal.
People can send back an **SOS**, an **"I'm safe"** or a **hazard report** the same way.

This notebook runs the same rules as the iPhone app, ported to Python in `alertmesh/`.
The warning bytes match the app's byte for byte (see `tests/`). What the notebook adds
is a simulation: phones on a map, passing warnings along.

**How to run:** open this file in VS Code and choose **Run All**. It takes about a minute.
Every run gives the same numbers, because the simulations use fixed random seeds.

*Simulation, not field data.* Bluetooth is modelled as a plain 60 m range with no signal
loss, and the internet as instant. Real Bluetooth reach varies from about 10 m to over 100 m.

In [ ]:
import pandas as pd

from alertmesh import geohash, metrics, places, proximity, reports, wire
from alertmesh.alert_store import AlertStore
from alertmesh.mesh_sim import COMMUNITY_REPORT_TYPE, OFFICIAL_ALERT_TYPE, Mesh, offset_m
from alertmesh.signer import OfficialAlertSigner, WarningDraft, new_alert_id
from alertmesh.wire import HazardType, Severity

START = metrics.START_MS            # simulation clock, in milliseconds
KATHERINE = places.find("Katherine")
print(f"{KATHERINE.name}: {KATHERINE.latitude}, {KATHERINE.longitude}")
print(places.CREDIT)

## 1. Issue a signed warning

The warning console (on a Mac in the real app) writes a warning and **signs** it with the
publisher's private key. Phones only trust warnings signed by the one key built into the app.
This demo uses the public *development* key, so anyone can reproduce it; a real deployment
would use a key that never leaves the publisher.

The warning names the area it covers as up to four **area codes** (geohashes). A 5-character
code is a box about 5 km across.

In [ ]:
area = geohash.encode(KATHERINE.latitude, KATHERINE.longitude, 5)
draft = WarningDraft(
    hazard=HazardType.BUSHFIRE,
    severity=Severity.EMERGENCY_WARNING,
    headline="Bushfire near Katherine - leave now",
    action_text="Travel north on the Stuart Highway. Do not wait.",
    duration_hours=6,
    area_cells=[area],
)
print("Problems with the draft:", draft.problems or "none")

signer = OfficialAlertSigner()          # development key
warning = signer.sign(draft, new_alert_id(), START)
payload = wire.encode(warning)

print(f"Area code: {area}  ({places.label(area)})")
print(f"Signed warning: {len(payload)} bytes (must fit one Bluetooth frame: {wire.MAX_ENCODED_BYTES} bytes)")
print("First bytes:", payload[:24].hex(" "))
print("Signed by the key phones trust?", wire.verify_pinned(warning))

The whole warning fits in **one Bluetooth frame**, which matters on a crowded or patchy
network. The signature is the only thing a phone with no internet can check, and it is enough:
nobody without the publisher's key can make a warning that passes.

## 2. A forged or edited warning goes nowhere

Two attacks a prankster might try:

- **Forge** a warning with their own key ("Fake fire, drive south").
- **Edit** a real one: downgrade an Emergency Warning to Advice, or move its area.

Both fail the signature check. In the mesh, the first honest phone to receive one throws
it away and passes nothing on, even if the sender's phone has been modified to flood it.

In [ ]:
from cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey
from dataclasses import replace

attacker = OfficialAlertSigner(Ed25519PrivateKey.generate().private_bytes_raw())
forged = attacker.sign(replace(draft, headline="Fake fire - drive south now"), new_alert_id(), START)
downgraded = replace(warning, severity=Severity.ADVICE)
moved = replace(warning, area_cells=(geohash.encode(-12.4634, 130.8456, 5),))   # to Darwin

store = AlertStore(clock=lambda: START)
pd.DataFrame(
    [
        (name, wire.verify_pinned(item), store.ingest(item).value)
        for name, item in [("genuine", warning), ("forged", forged), ("downgraded", downgraded), ("moved", moved)]
    ],
    columns=["warning", "signature valid", "phone's store says"],
)

In [ ]:
# A line of 5 phones, 50 m apart. Phone 0 has been modified to flood a forgery anyway.
line = Mesh(start_ms=START)
phones = [line.add_phone(f"p{i}", *offset_m(KATHERINE.latitude, KATHERINE.longitude, 0, 50 * i)) for i in range(5)]
reached = line.broadcast(phones[0], OFFICIAL_ALERT_TYPE, wire.encode(forged))
print(f"Forgery reached {reached} phone(s); phones now holding it: {sum(bool(p.alert_store.live_alerts()) for p in phones)}")

line.send(phones[0], OFFICIAL_ALERT_TYPE, payload)
print(f"Genuine warning: phones holding it: {sum(bool(p.alert_store.live_alerts()) for p in phones)} of 5")

## 3. Watch the warning spread

A community of **300 phones** spread over about 1.2 km × 1.2 km around Katherine.
Only **1 in 10** has internet at the moment. A quarter of the people are walking about.

The warning is published at minute 0. Phones with internet get it at once and hand it to
the phones next to them over Bluetooth, which pass it on (up to 7 hops). After that,
phones swap what they hold whenever they meet, so people walking about carry it to
groups that were out of reach.

Press **▶** under the map to play it. Blue: warned over the internet. Red: warned over
Bluetooth. Grey: not warned yet. The whole map is inside the warning area.

In [ ]:
from alertmesh import viz

town = metrics.Scenario(n_phones=300, area_m=1200, share_online=0.10, share_moving=0.25, speed_mps=1.4, seed=4)
town_mesh, town_warning = metrics.build(town)
town_mesh.publish(wire.encode(town_warning))
frames = viz.spread_frames(town_mesh, town_warning.alert_id, minutes=30)

viz.spread_map(frames, title="Bushfire warning spreading through Katherine (simulation)")

In [ ]:
shares = viz.share_by_minute(frames).set_index("minute")["share warned"]
pd.DataFrame({"share of phones warned": shares.loc[[0, 1, 2, 5, 10, 15, 20, 30]].map("{:.0%}".format)})

Straight away about half the phones have it: the 1 in 10 with internet, plus everyone
within 7 Bluetooth hops of them. The rest arrive over the next minutes as people walk into
range of someone who already has it. Without the mesh, only the phones with internet would
ever be warned (section 8 compares the two).